# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahmanislamzada/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task Type: Regression / Scoring (Ranking Signal Analysis)

Framing: Predicting total GSC search clicks (gsc_clicks) for individual content pages based on historical position, impressions, and volatility metrics.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

Target Variable: Total GSC Clicks (total_clicks aggregated at page level over time).

Why: Clicks serve as a direct proxy for high-converting organic search visibility and content engagement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

Metric: $R^2$ Score & Root Mean Squared Error (RMSE) on GroupShuffleSplit (per-client test set).Threshold: Achieving $R^2 > 0.60$ across unseen client domains.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect(':memory:')

# Try loading existing data or generate fallback sample
data_loaded = False
for file_path in ["data/sample_w01.csv", "../data/sample_w01.csv", "data/*.parquet", "../data/*.parquet"]:
    try:
        if file_path.endswith('.csv'):
            con.execute(f"CREATE TABLE fact_daily AS SELECT * FROM read_csv_auto('{file_path}')")
        else:
            con.execute(f"CREATE TABLE fact_daily AS SELECT * FROM read_parquet('{file_path}')")
        data_loaded = True
        break
    except Exception:
        continue

if not data_loaded:
    np.random.seed(42)
    n = 500
    df_sample = pd.DataFrame({
        'content_hash_id': [f"page_{i%50}" for i in range(n)],
        'client_hash_id': [f"client_{i%5}" for i in range(n)],
        'gsc_clicks': np.random.poisson(lam=15, size=n),
        'gsc_impressions': np.random.poisson(lam=300, size=n),
        'gsc_sum_position': np.random.uniform(500, 3000, size=n)
    })
    con.register('df_sample', df_sample)
    con.execute("CREATE TABLE fact_daily AS SELECT * FROM df_sample")

# Aggregate to show Unit of Analysis: 1 Row = 1 Page (content_hash_id per client)
df_unit = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS target_total_clicks,
        SUM(gsc_impressions) AS total_impressions,
        AVG(gsc_clicks::FLOAT / NULLIF(gsc_impressions, 0)) AS avg_ctr,
        AVG(gsc_sum_position::FLOAT / NULLIF(gsc_impressions, 0)) AS avg_position
    FROM fact_daily
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 50
    LIMIT 5
""").df()

print("--- UNIT OF ANALYSIS DATAFRAME (1 Row = 1 Unique Page) ---")
display(df_unit)

--- UNIT OF ANALYSIS DATAFRAME (1 Row = 1 Unique Page) ---


,client_hash_id,content_hash_id,target_total_clicks,total_impressions,avg_ctr,avg_position
0,client_0,page_0,176.0,3026.0,0.058451,4.946722
1,client_1,page_1,164.0,2973.0,0.055247,5.549971
2,client_2,page_2,167.0,3000.0,0.055512,6.344598
3,client_3,page_3,149.0,2997.0,0.049843,5.397840
4,client_4,page_4,129.0,3031.0,0.042573,6.456964


## 5. Why ML beats a fixed rule here

Why ML: Search algorithms exhibit non-linear interactions between CTR, keyword volatility, and ranking positions. A heuristic rule (e.g., "if position < 5, update content") fails to account for client-specific baselines and traffic volume scales. A trained tree-based model captures these non-linear feature combinations automatically.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.